# T-SQL Notes: Vertical Combination of Queries (Set Operators) & Filtering (WHERE)

**Topics covered**

1. Relational model & set theory (F. Codd)
2. Set operators: `UNION ALL`, `UNION`, `INTERSECT`, `EXCEPT`, and `SELECT ... INTO`
3. Exercises on **Northwind** (cities shared by customers and employees)
4. Written order vs. logical order of a `SELECT` statement
5. Filtering with `WHERE`: comparison operators, `AND / OR / NOT`, `BETWEEN`, `IN`, `IS NULL`, `LIKE`
6. Performance rule: *SARGable* predicates
7. Pattern matching on **AdventureWorksDW2022**
8. Cheat sheet

> **How this notebook works.** The Jupyter kernel is Python, so the SQL Server queries cannot be executed here.
> Each SQL query is stored inside a call to a small helper, `show(...)`, which prints the **expected output**
> as it would appear in the SSMS results grid. Running a cell simply re-prints that stored output.
>
> **Note on the data.** Outputs were produced from the standard Microsoft *pubs* sample database
> (and from the well-known Northwind data). If your instructor's copy was modified, row counts may differ slightly.

In [1]:
import pandas as pd

def show(sql, cols=None, rows=None, total=None, message=None):
    """Pretty-prints the expected output of a T-SQL query (hard-coded, no database needed)."""
    if cols is not None:
        df = pd.DataFrame(rows, columns=cols).fillna("NULL")
        print(df.to_string(index=False))
        if total:
            print(f"\n... {total} rows in total ({len(df)} shown)")
        else:
            print(f"\n({len(df)} rows affected)")
    if message:
        print(("\n" if cols is not None else "") + message)

---
# Part 1 - Relational Model and Set Theory

**E. F. Codd** (1970) described the **relational model**: data is stored in *relations* (tables), and a table is a **set of rows**.
Because a table is a set, we can apply the classical set operations to the *results of queries*:

| Set operation | Symbol | T-SQL operator | Duplicates in result |
|---|---|---|---|
| Union (keeping duplicates) | A + B | `UNION ALL` | kept |
| Union | A ∪ B | `UNION` | removed |
| Intersection | A ∩ B | `INTERSECT` | removed |
| Difference | A - B = A - (A ∩ B) | `EXCEPT` | removed |

These operators combine tables **vertically**: rows of one query are stacked under the rows of another.
(`JOIN`, by contrast, combines tables *horizontally* by placing columns side by side.)

### The same idea in plain Python

In [2]:
A = {10, 50, 30}
B = {90, 20, 30, 50}

print("A ∩ B  (INTERSECT) :", sorted(A & B))
print("A - B  (EXCEPT)    :", sorted(A - B))
print("A ∪ B  (UNION)     :", sorted(A | B))
print("A + B  (UNION ALL) :", sorted(list(A) + list(B)))

A ∩ B  (INTERSECT) : [30, 50]
A - B  (EXCEPT)    : [10]
A ∪ B  (UNION)     : [10, 20, 30, 50, 90]
A + B  (UNION ALL) : [10, 20, 30, 30, 50, 50, 90]


### Rules that apply to *all* set operators

1. Every query must return the **same number of columns**.
2. Corresponding columns (1st with 1st, 2nd with 2nd, ...) must have **compatible data types**.
3. **Column names** of the final result come from the **first** query only (aliases in later queries are ignored).
4. If a query lacks a column, supply a placeholder such as `NULL` (best: `CAST(NULL AS char(5))`).
5. `ORDER BY` is allowed **only once, at the very end** of the whole combined statement.
6. For `UNION`, `INTERSECT` and `EXCEPT`, two `NULL`s are treated as **equal** (unlike `=` in a `WHERE` clause).
7. Precedence: `INTERSECT` is evaluated **before** `UNION` and `EXCEPT`; otherwise operators are evaluated left to right. Use parentheses to be explicit.

## 1.1 `UNION ALL`

**Purpose:** stack the rows of two or more queries **without** removing duplicates. It is the fastest operator because SQL Server does not need to compare rows.

**General form**

```sql
SELECT col1, col2, ... FROM table_a
UNION ALL
SELECT col1, col2, ... FROM table_b
[UNION ALL SELECT ... ]
[ORDER BY ...];
```

**Example 1 - stores + authors** (state, city, zip)

In [3]:
show(
    """
    select s.[state], s.city, s.zip from dbo.stores  as s   -- B
    union all
    select a.[state], a.city, a.zip from dbo.authors as a   -- C
    order by 1, 2;
    """,
    cols=['state', 'city', 'zip'],
    rows=[
        ('CA', 'Berkeley', '94705'),
        ('CA', 'Berkeley', '94705'),
        ('CA', 'Covelo', '95428'),
        ('CA', 'Fremont', '90019'),
        ('CA', 'Los Gatos', '96745'),
        ('CA', 'Menlo Park', '94025'),
        ('CA', 'Oakland', '94618'),
        ('CA', 'Oakland', '94609'),
    ],
    total=29,
)

state       city   zip
   CA   Berkeley 94705
   CA   Berkeley 94705
   CA     Covelo 95428
   CA    Fremont 90019
   CA  Los Gatos 96745
   CA Menlo Park 94025
   CA    Oakland 94618
   CA    Oakland 94609

... 29 rows in total (8 shown)


**Example 2 - the instructor's three-table version** (publishers have no ZIP column, so `NULL` is used)

> Typo fix: the original script starts with `dunion all`. It must be `union all`.

In [4]:
show(
    """
    select p.[state], p.city, NULL as zip from dbo.publishers as p   -- A
    union all
    select s.[state], s.city, s.zip       from dbo.stores     as s   -- B
    union all
    select a.[state], a.city, a.zip       from dbo.authors    as a   -- C
    """,
    cols=['state', 'city', 'zip'],
    rows=[
        ('MA', 'Boston', None),
        ('DC', 'Washington', None),
        ('CA', 'Berkeley', None),
        ('IL', 'Chicago', None),
        ('TX', 'Dallas', None),
        (None, 'München', None),
        ('NY', 'New York', None),
        (None, 'Paris', None),
    ],
    total=37,
)

state       city  zip
   MA     Boston NULL
   DC Washington NULL
   CA   Berkeley NULL
   IL    Chicago NULL
   TX     Dallas NULL
 NULL    München NULL
   NY   New York NULL
 NULL      Paris NULL

... 37 rows in total (8 shown)


## 1.2 `UNION`

**Purpose:** same as `UNION ALL`, but **duplicate rows are removed** (it behaves like a `DISTINCT` over the whole result).
SQL Server must sort or hash the rows to find duplicates, so it is slower than `UNION ALL`.
Use `UNION ALL` whenever duplicates are impossible or acceptable.

**General form**

```sql
SELECT ... FROM table_a
UNION
SELECT ... FROM table_b;
```

**Example 3 - distinct locations from all three tables**

In [5]:
show(
    """
    select p.[state], p.city, NULL as zip      from dbo.publishers as p   -- A
    union
    select s.[state], s.city, s.zip as Shalgham from dbo.stores    as s   -- B  (alias is ignored!)
    union
    select a.[state], a.city, a.zip            from dbo.authors   as a   -- C
    """,
    cols=['state', 'city', 'zip'],
    rows=[
        (None, 'München', None),
        (None, 'Paris', None),
        ('CA', 'Berkeley', None),
        ('CA', 'Berkeley', '94705'),
        ('CA', 'Covelo', '95428'),
        ('CA', 'Fremont', '90019'),
        ('CA', 'Los Gatos', '96745'),
        ('CA', 'Menlo Park', '94025'),
    ],
    total=31,
)

state       city   zip
 NULL    München  NULL
 NULL      Paris  NULL
   CA   Berkeley  NULL
   CA   Berkeley 94705
   CA     Covelo 95428
   CA    Fremont 90019
   CA  Los Gatos 96745
   CA Menlo Park 94025

... 31 rows in total (8 shown)


Notice that the output column is called `zip`, **not** `Shalgham`: names come from the first query.

**Example 4 - `UNION` removes the duplicates that `UNION ALL` keeps** (Oakland has four authors with the same ZIP)

In [6]:
show(
    """
    select 'UNION ALL' as operator, count(*) as row_count
    from ( select a.[state], a.city, a.zip from dbo.authors as a
           union all
           select a.[state], a.city, a.zip from dbo.authors as a ) as x
    union all
    select 'UNION', count(*)
    from ( select a.[state], a.city, a.zip from dbo.authors as a
           union
           select a.[state], a.city, a.zip from dbo.authors as a ) as x;
    """,
    cols=['operator', 'row_count'],
    rows=[
        ('UNION ALL', 46),
        ('UNION', 17),
    ],
)

 operator  row_count
UNION ALL         46
    UNION         17

(2 rows affected)


## 1.3 Making-table query: `SELECT ... INTO`

**Purpose:** create a **new table** and fill it with the result of a query in a single step (DDL + DML).

**General form**

```sql
SELECT col1, col2, ...
INTO   new_table_name          -- placed in the FIRST query only
FROM   source_table
[UNION | UNION ALL | INTERSECT | EXCEPT
 SELECT ... FROM another_table];
```

* The table **must not already exist**, otherwise you get an error.
* Column **names and data types** are taken from the **first** query.
* Constraints, indexes and keys are **not** copied.
* To get a precise type for a placeholder, use `CAST(NULL AS char(5)) AS zip`.

**Example 5 - build a `Locations` table**

In [7]:
show(
    """
    select p.[state], p.city, CAST(NULL as char(5)) as zip
    into   dbo.Locations
    from   dbo.publishers as p
    union
    select s.[state], s.city, s.zip from dbo.stores  as s
    union
    select a.[state], a.city, a.zip from dbo.authors as a;
    """,
  
)

In [8]:
show(
    """
    select * from dbo.Locations order by [state], city;
    """,
    cols=['state', 'city', 'zip'],
    rows=[
        (None, 'München', None),
        (None, 'Paris', None),
        ('CA', 'Berkeley', None),
        ('CA', 'Berkeley', '94705'),
        ('CA', 'Covelo', '95428'),
        ('CA', 'Fremont', '90019'),
        ('CA', 'Los Gatos', '96745'),
        ('CA', 'Menlo Park', '94025'),
    ],
    total=31,
)

state       city   zip
 NULL    München  NULL
 NULL      Paris  NULL
   CA   Berkeley  NULL
   CA   Berkeley 94705
   CA     Covelo 95428
   CA    Fremont 90019
   CA  Los Gatos 96745
   CA Menlo Park 94025

... 31 rows in total (8 shown)


## 1.4 `INTERSECT`

**Purpose:** return only the rows that appear in **both** queries (duplicates removed).
Typical question: *"Which items exist in both lists?"*

**General form**

```sql
SELECT col1, col2 FROM table_a
INTERSECT
SELECT col1, col2 FROM table_b;
```

**Example 6 - (state, city) pairs that have both a publisher and an author**

In [9]:
show(
    """
    select p.[state], p.city from dbo.publishers as p   -- A
    intersect
    select a.[state], a.city from dbo.authors    as a   -- B
    """,
    cols=['state', 'city'],
    rows=[
        ('CA', 'Berkeley'),
    ],
)

state     city
   CA Berkeley

(1 rows affected)


## 1.5 `EXCEPT`

**Purpose:** return the rows of the **first** query that do **not** appear in the second one (duplicates removed).
Mathematically: **A - B = A - (A ∩ B)**.
Typical question: *"What is in list A but missing from list B?"*

> **Order matters:** `A EXCEPT B` is not the same as `B EXCEPT A`.

**General form**

```sql
SELECT col1, col2 FROM table_a
EXCEPT
SELECT col1, col2 FROM table_b;
```

**Example 7 - cities with a publisher but no author**

In [10]:
show(
    """
    select p.[state], p.city from dbo.publishers as p   -- A
    except
    select a.[state], a.city from dbo.authors    as a   -- B
    order by 1, 2;
    """,
    cols=['state', 'city'],
    rows=[
        (None, 'München'),
        (None, 'Paris'),
        ('DC', 'Washington'),
        ('IL', 'Chicago'),
        ('MA', 'Boston'),
        ('NY', 'New York'),
        ('TX', 'Dallas'),
    ],
)

state       city
 NULL    München
 NULL      Paris
   DC Washington
   IL    Chicago
   MA     Boston
   NY   New York
   TX     Dallas

(7 rows affected)


**Example 8 - the reverse direction: cities with an author but no publisher**

The `authors` table has 23 rows, but because `EXCEPT` also removes duplicates, the result is much smaller.

In [11]:
show(
    """
    select a.[state], a.city from dbo.authors    as a   -- A
    except
    select p.[state], p.city from dbo.publishers as p   -- B
    order by 1, 2;
    """,
    cols=['state', 'city'],
    rows=[
        ('CA', 'Covelo'),
        ('CA', 'Menlo Park'),
        ('CA', 'Oakland'),
        ('CA', 'Palo Alto'),
        ('CA', 'San Francisco'),
        ('CA', 'San Jose'),
        ('CA', 'Vacaville'),
        ('CA', 'Walnut Creek'),
        ('IN', 'Gary'),
        ('KS', 'Lawrence'),
        ('MD', 'Rockville'),
        ('MI', 'Ann Arbor'),
        ('OR', 'Corvallis'),
        ('TN', 'Nashville'),
        ('UT', 'Salt Lake City'),
    ],
)

state           city
   CA         Covelo
   CA     Menlo Park
   CA        Oakland
   CA      Palo Alto
   CA  San Francisco
   CA       San Jose
   CA      Vacaville
   CA   Walnut Creek
   IN           Gary
   KS       Lawrence
   MD      Rockville
   MI      Ann Arbor
   OR      Corvallis
   TN      Nashville
   UT Salt Lake City

(15 rows affected)


## 1.6 Common error: different number of columns

In [12]:
show(
    """
    select a.[state], a.city, a.zip from dbo.authors as a
    union
    select s.[state], s.city        from dbo.stores  as s;
    """,
    message='Msg 205, Level 16, State 1, Line 3\nAll queries combined using a UNION, INTERSECT or EXCEPT operator must have an equal number of expressions in their target lists.'
)

Msg 205, Level 16, State 1, Line 3
All queries combined using a UNION, INTERSECT or EXCEPT operator must have an equal number of expressions in their target lists.


---
# Part 2 - The `SELECT` Statement: Written Order vs. Logical Order

**Order in which you WRITE the clauses** (the instructor's outline)

```sql
SELECT   [DISTINCT] [TOP (n)] column_list
FROM     table_name
WHERE    filter_condition          -- <<< today's topic
ORDER BY column_list;
```

**Order in which SQL Server LOGICALLY processes them**

| Step | Clause | What it does |
|---|---|---|
| 1 | `FROM` | choose the source table(s) |
| 2 | `WHERE` | keep only rows that satisfy the condition |
| 3 | `GROUP BY` / `HAVING` | grouping and group filters (later topics) |
| 4 | `SELECT` | compute the output columns and aliases |
| 5 | `DISTINCT` | remove duplicate rows |
| 6 | `ORDER BY` | sort the result |
| 7 | `TOP` | keep the first n rows |

Consequence: a column **alias** defined in `SELECT` cannot be used in `WHERE` (it does not exist yet), but it **can** be used in `ORDER BY`.

---
# Part 3 - Filtering Rows with `WHERE`

**Purpose:** keep only the rows for which the condition evaluates to **TRUE**. Rows that evaluate to FALSE **or UNKNOWN (NULL)** are discarded.

**General form**

```sql
SELECT column_list
FROM   table_name
WHERE  <predicate> [AND | OR <predicate> ...];
```

## 3.1 Comparison operators

| Operator | Meaning | Python equivalent |
|---|---|---|
| `=` | equal | `==` |
| `<>` (or `!=`) | not equal | `!=` |
| `>`  `>=` | greater than / or equal | `>`  `>=` |
| `<`  `<=` | less than / or equal | `<`  `<=` |

**Example 9 - numeric comparison: books priced at 20 or more**

In [13]:
show(
    """
    use pubs
    go
    select t.title_id, t.title, t.price, t.pubdate
    from   dbo.titles as t
    where  t.price >= 20;
    """,
    cols=['title_id', 'title', 'price', 'pubdate'],
    rows=[
        ('PC1035', 'But Is It User Friendly?', '22.95', '1991-06-30'),
        ('PC8888', 'Secrets of Silicon Valley', '20.00', '1994-06-12'),
        ('PS1372', 'Computer Phobic AND Non-Phobic Indi', '21.59', '1991-10-21'),
        ('TC3218', 'Onions, Leeks, and Garlic: Cooking ', '20.95', '1991-10-21'),
    ],
)

title_id                               title price    pubdate
  PC1035            But Is It User Friendly? 22.95 1991-06-30
  PC8888           Secrets of Silicon Valley 20.00 1994-06-12
  PS1372 Computer Phobic AND Non-Phobic Indi 21.59 1991-10-21
  TC3218 Onions, Leeks, and Garlic: Cooking  20.95 1991-10-21

(4 rows affected)


**Example 10 - text equality** (string literals use single quotes)

In [14]:
show(
    """
    select t.title_id, t.title, t.price, t.pubdate, t.[type]
    from   dbo.titles as t
    where  t.[type] = 'Business';       -- Python: ==
    """,
    cols=['title_id', 'title', 'price', 'pubdate', 'type'],
    rows=[
        ('BU1032', "The Busy Executive's Database Guide", '19.99', '1991-06-12', 'business'),
        ('BU1111', 'Cooking with Computers: Surreptitio', '11.95', '1991-06-09', 'business'),
        ('BU2075', 'You Can Combat Computer Stress!', '2.99', '1991-06-30', 'business'),
        ('BU7832', 'Straight Talk About Computers', '19.99', '1991-06-22', 'business'),
    ],
)

title_id                               title price    pubdate     type
  BU1032 The Busy Executive's Database Guide 19.99 1991-06-12 business
  BU1111 Cooking with Computers: Surreptitio 11.95 1991-06-09 business
  BU2075     You Can Combat Computer Stress!  2.99 1991-06-30 business
  BU7832       Straight Talk About Computers 19.99 1991-06-22 business

(4 rows affected)


**Example 11 - not equal (`<>`)**

In [15]:
show(
    """
    select t.title_id, t.title, t.price, t.pubdate, t.[type]
    from   dbo.titles as t
    where  t.[type] <> 'Business';
    """,
    cols=['title_id', 'title', 'price', 'pubdate', 'type'],
    rows=[
        ('MC2222', 'Silicon Valley Gastronomic Treats', '19.99', '1991-06-09', 'mod_cook'),
        ('MC3021', 'The Gourmet Microwave', '2.99', '1991-06-18', 'mod_cook'),
        ('MC3026', 'The Psychology of Computer Cooking', None, '1991-06-12', 'UNDECIDED'),
        ('PC1035', 'But Is It User Friendly?', '22.95', '1991-06-30', 'popular_comp'),
        ('PC8888', 'Secrets of Silicon Valley', '20.00', '1994-06-12', 'popular_comp'),
        ('PC9999', 'Net Etiquette', None, '1991-06-12', 'popular_comp'),
        ('PS1372', 'Computer Phobic AND Non-Phobic Indi', '21.59', '1991-10-21', 'psychology'),
        ('PS2091', 'Is Anger the Enemy?', '10.95', '1991-06-15', 'psychology'),
    ],
    total=14,
)

title_id                               title price    pubdate         type
  MC2222   Silicon Valley Gastronomic Treats 19.99 1991-06-09     mod_cook
  MC3021               The Gourmet Microwave  2.99 1991-06-18     mod_cook
  MC3026  The Psychology of Computer Cooking  NULL 1991-06-12    UNDECIDED
  PC1035            But Is It User Friendly? 22.95 1991-06-30 popular_comp
  PC8888           Secrets of Silicon Valley 20.00 1994-06-12 popular_comp
  PC9999                       Net Etiquette  NULL 1991-06-12 popular_comp
  PS1372 Computer Phobic AND Non-Phobic Indi 21.59 1991-10-21   psychology
  PS2091                 Is Anger the Enemy? 10.95 1991-06-15   psychology

... 14 rows in total (8 shown)


## 3.2 Logical operators: `AND`, `OR`, `NOT`

| Operator | Result is TRUE when ... |
|---|---|
| `AND` | **both** conditions are true |
| `OR` | **at least one** condition is true |
| `NOT` | the condition is false |

Precedence: `NOT` > `AND` > `OR`. Use parentheses whenever you mix `AND` with `OR`.

## 3.3 `BETWEEN ... AND ...`

**Purpose:** test whether a value lies in a range, **both ends inclusive**.
`x BETWEEN a AND b` is the same as `x >= a AND x <= b`. The negation is `NOT BETWEEN`.

**Implicit conversion:** the text `'1991-01-01'` is automatically converted (implicitly cast) to a `datetime` because it is compared with a date column.

**Example 12 - not Business, published in the first half of 1991**

In [17]:
show(
    """
    select t.title_id, t.title, t.price, t.pubdate, t.[type]
    from   dbo.titles as t
    where  t.[type] <> 'Business'
      and  t.pubdate between '1991-01-01' and '1991-06-30';
    """,
    cols=['title_id', 'title', 'price', 'pubdate', 'type'],
    rows=[
        ('MC2222', 'Silicon Valley Gastronomic Tre', '19.99', '1991-06-09', 'mod_cook'),
        ('MC3021', 'The Gourmet Microwave', '2.99', '1991-06-18', 'mod_cook'),
        ('MC3026', 'The Psychology of Computer Coo', None, '1991-06-12', 'UNDECIDED'),
        ('PC1035', 'But Is It User Friendly?', '22.95', '1991-06-30', 'popular_comp'),
        ('PC9999', 'Net Etiquette', None, '1991-06-12', 'popular_comp'),
        ('PS2091', 'Is Anger the Enemy?', '10.95', '1991-06-15', 'psychology'),
        ('PS3333', 'Prolonged Data Deprivation: Fo', '19.99', '1991-06-12', 'psychology'),
        ('PS7777', 'Emotional Security: A New Algo', '7.99', '1991-06-12', 'psychology'),
        ('TC4203', 'Fifty Years in Buckingham Pala', '11.95', '1991-06-12', 'trad_cook'),
        ('TC7777', 'Sushi, Anyone?', '14.99', '1991-06-12', 'trad_cook'),
    ],
)

title_id                          title price    pubdate         type
  MC2222 Silicon Valley Gastronomic Tre 19.99 1991-06-09     mod_cook
  MC3021          The Gourmet Microwave  2.99 1991-06-18     mod_cook
  MC3026 The Psychology of Computer Coo  NULL 1991-06-12    UNDECIDED
  PC1035       But Is It User Friendly? 22.95 1991-06-30 popular_comp
  PC9999                  Net Etiquette  NULL 1991-06-12 popular_comp
  PS2091            Is Anger the Enemy? 10.95 1991-06-15   psychology
  PS3333 Prolonged Data Deprivation: Fo 19.99 1991-06-12   psychology
  PS7777 Emotional Security: A New Algo  7.99 1991-06-12   psychology
  TC4203 Fifty Years in Buckingham Pala 11.95 1991-06-12    trad_cook
  TC7777                 Sushi, Anyone? 14.99 1991-06-12    trad_cook

(10 rows affected)


**Example 13 - `NOT BETWEEN`** (everything outside that period)

In [18]:
show(
    """
    select t.title_id, t.title, t.price, t.pubdate, t.[type]
    from   dbo.titles as t
    where  t.[type] <> 'Business'
      and  t.pubdate not between '1991-01-01' and '1991-06-30';
    """,
    cols=['title_id', 'title', 'price', 'pubdate', 'type'],
    rows=[
        ('PC8888', 'Secrets of Silicon Valley', '20.00', '1994-06-12', 'popular_comp'),
        ('PS1372', 'Computer Phobic AND Non-Phobic', '21.59', '1991-10-21', 'psychology'),
        ('PS2106', 'Life Without Fear', '7.00', '1991-10-05', 'psychology'),
        ('TC3218', 'Onions, Leeks, and Garlic: Coo', '20.95', '1991-10-21', 'trad_cook'),
    ],
)

title_id                          title price    pubdate         type
  PC8888      Secrets of Silicon Valley 20.00 1994-06-12 popular_comp
  PS1372 Computer Phobic AND Non-Phobic 21.59 1991-10-21   psychology
  PS2106              Life Without Fear  7.00 1991-10-05   psychology
  TC3218 Onions, Leeks, and Garlic: Coo 20.95 1991-10-21    trad_cook

(4 rows affected)


> **Careful with `datetime` columns.** If `pubdate` contains a time part (for example `1991-06-30 14:00`),
> `BETWEEN ... AND '1991-06-30'` silently **misses** that row, because `'1991-06-30'` means midnight.
> A safer, half-open form is `pubdate >= '1991-01-01' AND pubdate < '1991-07-01'`.

## 3.4 The Golden Rule: write *SARGable* predicates

**SARG** = *Search ARGument able*. A predicate is SARGable when SQL Server can use an **index seek** on the column.
Wrapping the **column** in a function (`YEAR(col)`, `MONTH(col)`, `UPPER(col)`, ...) forces SQL Server to evaluate the function for **every row** (an index/table *scan*), which hurts performance on big tables.

| Slow (non-SARGable) | Fast (SARGable) |
|---|---|
| `WHERE YEAR(pubdate) = 1991 AND MONTH(pubdate) BETWEEN 1 AND 6` | `WHERE pubdate >= '1991-01-01' AND pubdate < '1991-07-01'` |

Both queries return the same rows, but the second one lets an index on `pubdate` be used.

In [19]:
show(
    """
    -- Non-SARGable: function applied to the column
    select t.title_id, t.title, t.price, t.pubdate, t.[type]
    from   dbo.titles as t
    where  t.[type] <> 'Business'
      and  YEAR(t.pubdate) = 1991 and MONTH(t.pubdate) between 1 and 6;
    """,
    cols=['title_id', 'title', 'price', 'pubdate', 'type'],
    rows=[
        ('MC2222', 'Silicon Valley Gastronomic Tre', '19.99', '1991-06-09', 'mod_cook'),
        ('MC3021', 'The Gourmet Microwave', '2.99', '1991-06-18', 'mod_cook'),
        ('MC3026', 'The Psychology of Computer Coo', None, '1991-06-12', 'UNDECIDED'),
        ('PC1035', 'But Is It User Friendly?', '22.95', '1991-06-30', 'popular_comp'),
        ('PC9999', 'Net Etiquette', None, '1991-06-12', 'popular_comp'),
        ('PS2091', 'Is Anger the Enemy?', '10.95', '1991-06-15', 'psychology'),
        ('PS3333', 'Prolonged Data Deprivation: Fo', '19.99', '1991-06-12', 'psychology'),
        ('PS7777', 'Emotional Security: A New Algo', '7.99', '1991-06-12', 'psychology'),
        ('TC4203', 'Fifty Years in Buckingham Pala', '11.95', '1991-06-12', 'trad_cook'),
        ('TC7777', 'Sushi, Anyone?', '14.99', '1991-06-12', 'trad_cook'),
    ],
)

title_id                          title price    pubdate         type
  MC2222 Silicon Valley Gastronomic Tre 19.99 1991-06-09     mod_cook
  MC3021          The Gourmet Microwave  2.99 1991-06-18     mod_cook
  MC3026 The Psychology of Computer Coo  NULL 1991-06-12    UNDECIDED
  PC1035       But Is It User Friendly? 22.95 1991-06-30 popular_comp
  PC9999                  Net Etiquette  NULL 1991-06-12 popular_comp
  PS2091            Is Anger the Enemy? 10.95 1991-06-15   psychology
  PS3333 Prolonged Data Deprivation: Fo 19.99 1991-06-12   psychology
  PS7777 Emotional Security: A New Algo  7.99 1991-06-12   psychology
  TC4203 Fifty Years in Buckingham Pala 11.95 1991-06-12    trad_cook
  TC7777                 Sushi, Anyone? 14.99 1991-06-12    trad_cook

(10 rows affected)


In [20]:
show(
    """
    -- SARGable: the column is left untouched
    select t.title_id, t.title, t.price, t.pubdate, t.[type]
    from   dbo.titles as t
    where  t.[type] <> 'Business'
      and  t.pubdate >= '1991-01-01' and t.pubdate < '1991-07-01';
    """,
    cols=['title_id', 'title', 'price', 'pubdate', 'type'],
    rows=[
        ('MC2222', 'Silicon Valley Gastronomic Tre', '19.99', '1991-06-09', 'mod_cook'),
        ('MC3021', 'The Gourmet Microwave', '2.99', '1991-06-18', 'mod_cook'),
        ('MC3026', 'The Psychology of Computer Coo', None, '1991-06-12', 'UNDECIDED'),
        ('PC1035', 'But Is It User Friendly?', '22.95', '1991-06-30', 'popular_comp'),
        ('PC9999', 'Net Etiquette', None, '1991-06-12', 'popular_comp'),
        ('PS2091', 'Is Anger the Enemy?', '10.95', '1991-06-15', 'psychology'),
        ('PS3333', 'Prolonged Data Deprivation: Fo', '19.99', '1991-06-12', 'psychology'),
        ('PS7777', 'Emotional Security: A New Algo', '7.99', '1991-06-12', 'psychology'),
        ('TC4203', 'Fifty Years in Buckingham Pala', '11.95', '1991-06-12', 'trad_cook'),
        ('TC7777', 'Sushi, Anyone?', '14.99', '1991-06-12', 'trad_cook'),
    ],
)

title_id                          title price    pubdate         type
  MC2222 Silicon Valley Gastronomic Tre 19.99 1991-06-09     mod_cook
  MC3021          The Gourmet Microwave  2.99 1991-06-18     mod_cook
  MC3026 The Psychology of Computer Coo  NULL 1991-06-12    UNDECIDED
  PC1035       But Is It User Friendly? 22.95 1991-06-30 popular_comp
  PC9999                  Net Etiquette  NULL 1991-06-12 popular_comp
  PS2091            Is Anger the Enemy? 10.95 1991-06-15   psychology
  PS3333 Prolonged Data Deprivation: Fo 19.99 1991-06-12   psychology
  PS7777 Emotional Security: A New Algo  7.99 1991-06-12   psychology
  TC4203 Fifty Years in Buckingham Pala 11.95 1991-06-12    trad_cook
  TC7777                 Sushi, Anyone? 14.99 1991-06-12    trad_cook

(10 rows affected)


## 3.5 `IN` and `NOT IN`

**Purpose:** compare a column with a **list** of values. `col IN (a, b, c)` is shorthand for `col = a OR col = b OR col = c`.

**General form**

```sql
WHERE column [NOT] IN (value1, value2, ...);
```

**Example 14 - authors from KS, MI or UT (long form with `OR`)**

In [21]:
show(
    """
    select a.au_id,
           CONCAT_WS(' ', a.au_fname, a.au_lname) as FullName,
           a.[state], a.city
    from   dbo.authors as a
    where  a.[state] = 'KS' or a.[state] = 'MI' or a.[state] = 'UT';
    """,
    cols=['au_id', 'FullName', 'state', 'city'],
    rows=[
        ('341-22-1782', 'Meander Smith', 'KS', 'Lawrence'),
        ('712-45-1867', 'Innes del Castillo', 'MI', 'Ann Arbor'),
        ('899-46-2035', 'Anne Ringer', 'UT', 'Salt Lake City'),
        ('998-72-3567', 'Albert Ringer', 'UT', 'Salt Lake City'),
    ],
)

      au_id           FullName state           city
341-22-1782      Meander Smith    KS       Lawrence
712-45-1867 Innes del Castillo    MI      Ann Arbor
899-46-2035        Anne Ringer    UT Salt Lake City
998-72-3567      Albert Ringer    UT Salt Lake City

(4 rows affected)


(`CONCAT_WS(separator, a, b, ...)` joins strings with a separator and **skips NULLs**.)

**Example 15 - the same result with `IN`**

In [22]:
show(
    """
    select a.au_id,
           CONCAT_WS(' ', a.au_fname, a.au_lname) as FullName,
           a.[state], a.city
    from   dbo.authors as a
    where  a.[state] IN ('KS', 'MI', 'UT');
    """,
    cols=['au_id', 'FullName', 'state', 'city'],
    rows=[
        ('341-22-1782', 'Meander Smith', 'KS', 'Lawrence'),
        ('712-45-1867', 'Innes del Castillo', 'MI', 'Ann Arbor'),
        ('899-46-2035', 'Anne Ringer', 'UT', 'Salt Lake City'),
        ('998-72-3567', 'Albert Ringer', 'UT', 'Salt Lake City'),
    ],
)

      au_id           FullName state           city
341-22-1782      Meander Smith    KS       Lawrence
712-45-1867 Innes del Castillo    MI      Ann Arbor
899-46-2035        Anne Ringer    UT Salt Lake City
998-72-3567      Albert Ringer    UT Salt Lake City

(4 rows affected)


**Example 16 - `NOT IN`: everybody else**

In [23]:
show(
    """
    select a.au_id,
           CONCAT_WS(' ', a.au_fname, a.au_lname) as FullName,
           a.[state], a.city
    from   dbo.authors as a
    where  a.[state] NOT IN ('KS', 'MI', 'UT');
    """,
    cols=['au_id', 'FullName', 'state', 'city'],
    rows=[
        ('172-32-1176', 'Johnson White', 'CA', 'Menlo Park'),
        ('213-46-8915', 'Marjorie Green', 'CA', 'Oakland'),
        ('238-95-7766', 'Cheryl Carson', 'CA', 'Berkeley'),
        ('267-41-2394', "Michael O'Leary", 'CA', 'San Jose'),
        ('274-80-9391', 'Dean Straight', 'CA', 'Oakland'),
        ('409-56-7008', 'Abraham Bennet', 'CA', 'Berkeley'),
    ],
    total=19,
)

      au_id        FullName state       city
172-32-1176   Johnson White    CA Menlo Park
213-46-8915  Marjorie Green    CA    Oakland
238-95-7766   Cheryl Carson    CA   Berkeley
267-41-2394 Michael O'Leary    CA   San Jose
274-80-9391   Dean Straight    CA    Oakland
409-56-7008  Abraham Bennet    CA   Berkeley

... 19 rows in total (6 shown)


> **NULL trap:** if the list contains a `NULL` (for example from a subquery), `NOT IN` returns **no rows at all**, because every comparison becomes UNKNOWN.

## 3.6 `IS NULL` and `IS NOT NULL`

`NULL` means *unknown / missing*. It is **not** equal to anything, not even to another `NULL`,
so `price = NULL` never returns rows. You must use `IS NULL` / `IS NOT NULL`.

**Example 17 - books without a price**

In [24]:
show(
    """
    select t.title_id, t.title, t.price, t.pubdate, t.[type]
    from   dbo.titles as t
    where  t.price IS NULL;
    """,
    cols=['title_id', 'title', 'price', 'pubdate', 'type'],
    rows=[
        ('MC3026', 'The Psychology of Computer Coo', None, '1991-06-12', 'UNDECIDED'),
        ('PC9999', 'Net Etiquette', None, '1991-06-12', 'popular_comp'),
    ],
)

title_id                          title price    pubdate         type
  MC3026 The Psychology of Computer Coo  NULL 1991-06-12    UNDECIDED
  PC9999                  Net Etiquette  NULL 1991-06-12 popular_comp

(2 rows affected)


**Example 18 - books that do have a price**

In [25]:
show(
    """
    select t.title_id, t.title, t.price, t.pubdate, t.[type]
    from   dbo.titles as t
    where  t.price IS NOT NULL;
    """,
    cols=['title_id', 'title', 'price', 'pubdate', 'type'],
    rows=[
        ('BU1032', "The Busy Executive's Database ", '19.99', '1991-06-12', 'business'),
        ('BU1111', 'Cooking with Computers: Surrep', '11.95', '1991-06-09', 'business'),
        ('BU2075', 'You Can Combat Computer Stress', '2.99', '1991-06-30', 'business'),
        ('BU7832', 'Straight Talk About Computers', '19.99', '1991-06-22', 'business'),
        ('MC2222', 'Silicon Valley Gastronomic Tre', '19.99', '1991-06-09', 'mod_cook'),
    ],
    total=16,
)

title_id                          title price    pubdate     type
  BU1032 The Busy Executive's Database  19.99 1991-06-12 business
  BU1111 Cooking with Computers: Surrep 11.95 1991-06-09 business
  BU2075 You Can Combat Computer Stress  2.99 1991-06-30 business
  BU7832  Straight Talk About Computers 19.99 1991-06-22 business
  MC2222 Silicon Valley Gastronomic Tre 19.99 1991-06-09 mod_cook

... 16 rows in total (5 shown)


## 3.7 Pattern matching with `LIKE`

**Purpose:** search text by **pattern** instead of exact equality.

**General form**

```sql
WHERE column [NOT] LIKE 'pattern' [ESCAPE 'escape_char'];
```

**Wildcards**

| Wildcard | Meaning | Example | Matches |
|---|---|---|---|
| `%` | any string of 0 or more characters | `'a%'` | starts with *a* |
| `_` | exactly **one** character | `'a_n%'` | *a*, any char, *n*, anything |
| `[abc]` / `[a-g]` | one character from a set or range | `'[a-g]%'` | starts with a letter from a to g |
| `[^abc]` | one character **not** in the set | `'[^c,a,r]%'` | does not start with c, a, r (or a comma) |

* Case sensitivity depends on the **collation**. The default (`..._CI_...`) is **C**ase **I**nsensitive, so `'a%'` also matches `Albert`.
* Inside `[ ]` a comma is a literal character, so `[c,a,r]` is the same as `[car,]`. Writing `[car]` is cleaner.
* To search for a literal wildcard use `ESCAPE`: `WHERE col LIKE '%50\%%' ESCAPE '\'` finds text containing `50%`.
* A pattern with a leading `%` (`'%a'`, `'%vi%'`) cannot use an index seek, so it is slower on large tables.

**Example 19 - exact match (no wildcards)**

In [26]:
show(
    """
    select a.au_fname as FirstName, a.au_lname as LastName, a.city
    from   dbo.authors as a
    where  a.au_fname = 'Michael';
    """,
    cols=['FirstName', 'LastName', 'city'],
    rows=[
        ('Michael', "O'Leary", 'San Jose'),
    ],
)

FirstName LastName     city
  Michael  O'Leary San Jose

(1 rows affected)


**Example 20 - starts with `a`** (`'a%'`)

In [27]:
show(
    """
    select a.au_fname as FirstName, a.au_lname as LastName, a.city
    from   dbo.authors as a
    where  a.au_fname like 'a%';
    """,
    cols=['FirstName', 'LastName', 'city'],
    rows=[
        ('Abraham', 'Bennet', 'Berkeley'),
        ('Ann', 'Dull', 'Palo Alto'),
        ('Akiko', 'Yokomoto', 'Walnut Creek'),
        ('Anne', 'Ringer', 'Salt Lake City'),
        ('Albert', 'Ringer', 'Salt Lake City'),
    ],
)

FirstName LastName           city
  Abraham   Bennet       Berkeley
      Ann     Dull      Palo Alto
    Akiko Yokomoto   Walnut Creek
     Anne   Ringer Salt Lake City
   Albert   Ringer Salt Lake City

(5 rows affected)


**Example 21 - ends with `a`** (`'%a'`)

In [28]:
show(
    """
    select a.au_fname as FirstName, a.au_lname as LastName, a.city
    from   dbo.authors as a
    where  a.au_fname like '%a';
    """,
    cols=['FirstName', 'LastName', 'city'],
    rows=[
        ('Livia', 'Karsen', 'Oakland'),
        ('Sylvia', 'Panteley', 'Rockville'),
    ],
)

FirstName LastName      city
    Livia   Karsen   Oakland
   Sylvia Panteley Rockville

(2 rows affected)


**Example 22 - contains `vi`** (`'%vi%'`)

In [29]:
show(
    """
    select a.au_fname as FirstName, a.au_lname as LastName, a.city
    from   dbo.authors as a
    where  a.au_fname like '%vi%';
    """,
    cols=['FirstName', 'LastName', 'city'],
    rows=[
        ('Livia', 'Karsen', 'Oakland'),
        ('Sylvia', 'Panteley', 'Rockville'),
    ],
)

FirstName LastName      city
    Livia   Karsen   Oakland
   Sylvia Panteley Rockville

(2 rows affected)


**Example 23 - underscore: `a`, any single character, `n`, then anything** (`'a_n%'`)

In [30]:
show(
    """
    select a.au_fname as FirstName, a.au_lname as LastName, a.city
    from   dbo.authors as a
    where  a.au_fname like 'a_n%';
    """,
    cols=['FirstName', 'LastName', 'city'],
    rows=[
        ('Ann', 'Dull', 'Palo Alto'),
        ('Anne', 'Ringer', 'Salt Lake City'),
    ],
)

FirstName LastName           city
      Ann     Dull      Palo Alto
     Anne   Ringer Salt Lake City

(2 rows affected)


**Example 24 - range: first letter between a and g, any character, then `n`** (`'[a-g]_n%'`)

In [31]:
show(
    """
    select a.au_fname as FirstName, a.au_lname as LastName, a.city
    from   dbo.authors as a
    where  a.au_fname like '[a-g]_n%';
    """,
    cols=['FirstName', 'LastName', 'city'],
    rows=[
        ('Ann', 'Dull', 'Palo Alto'),
        ('Anne', 'Ringer', 'Salt Lake City'),
    ],
)

FirstName LastName           city
      Ann     Dull      Palo Alto
     Anne   Ringer Salt Lake City

(2 rows affected)


**Example 25 - `NOT LIKE`: exclude names whose first letter is c, a or r and third letter is e** (`'[c,a,r]_e%'`)

In [32]:
show(
    """
    select a.au_fname as FirstName, a.au_lname as LastName, a.city
    from   dbo.authors as a
    where  a.au_fname not like '[c,a,r]_e%';
    """,
    cols=['FirstName', 'LastName', 'city'],
    rows=[
        ('Johnson', 'White', 'Menlo Park'),
        ('Marjorie', 'Green', 'Oakland'),
        ('Michael', "O'Leary", 'San Jose'),
        ('Dean', 'Straight', 'Oakland'),
        ('Meander', 'Smith', 'Lawrence'),
        ('Abraham', 'Bennet', 'Berkeley'),
        ('Ann', 'Dull', 'Palo Alto'),
        ('Burt', 'Gringlesby', 'Covelo'),
        ('Charlene', 'Locksley', 'San Francisco'),
        ('Morningstar', 'Greene', 'Nashville'),
        ('Reginald', 'Blotchet-Halls', 'Corvallis'),
        ('Akiko', 'Yokomoto', 'Walnut Creek'),
        ('Innes', 'del Castillo', 'Ann Arbor'),
        ('Michel', 'DeFrance', 'Gary'),
        ('Dirk', 'Stringer', 'Oakland'),
        ('Stearns', 'MacFeather', 'Oakland'),
        ('Livia', 'Karsen', 'Oakland'),
        ('Sylvia', 'Panteley', 'Rockville'),
        ('Sheryl', 'Hunter', 'Palo Alto'),
        ('Heather', 'McBadden', 'Vacaville'),
        ('Anne', 'Ringer', 'Salt Lake City'),
        ('Albert', 'Ringer', 'Salt Lake City'),
    ],
)

  FirstName       LastName           city
    Johnson          White     Menlo Park
   Marjorie          Green        Oakland
    Michael        O'Leary       San Jose
       Dean       Straight        Oakland
    Meander          Smith       Lawrence
    Abraham         Bennet       Berkeley
        Ann           Dull      Palo Alto
       Burt     Gringlesby         Covelo
   Charlene       Locksley  San Francisco
Morningstar         Greene      Nashville
   Reginald Blotchet-Halls      Corvallis
      Akiko       Yokomoto   Walnut Creek
      Innes   del Castillo      Ann Arbor
     Michel       DeFrance           Gary
       Dirk       Stringer        Oakland
    Stearns     MacFeather        Oakland
      Livia         Karsen        Oakland
     Sylvia       Panteley      Rockville
     Sheryl         Hunter      Palo Alto
    Heather       McBadden      Vacaville
       Anne         Ringer Salt Lake City
     Albert         Ringer Salt Lake City

(22 rows affected)


**Example 26 - caret `^`: first character is anything *except* c, a, r (and comma), third letter is `e`** (`'[^c,a,r]_e%'`)

Compare with Example 25: `NOT LIKE '[c,a,r]_e%'` returns *all* rows that fail the pattern, whereas `LIKE '[^c,a,r]_e%'`
returns only rows that **match** the third-letter-*e* rule with a different first letter.

In [33]:
show(
    """
    select a.au_fname as FirstName, a.au_lname as LastName, a.city
    from   dbo.authors as a
    where  a.au_fname like '[^c,a,r]_e%';
    """,
    cols=['FirstName', 'LastName', 'city'],
    rows=[
        ('Stearns', 'MacFeather', 'Oakland'),
        ('Sheryl', 'Hunter', 'Palo Alto'),
    ],
)

FirstName   LastName      city
  Stearns MacFeather   Oakland
   Sheryl     Hunter Palo Alto

(2 rows affected)


---
# Part 4 - Pattern Matching on AdventureWorksDW2022

In the `DimProduct` table, `ProductAlternateKey` holds codes such as `BK-R93R-62` (bike, road, model 93, colour R, size 62).
Patterns are ideal for validating or searching this kind of structured code.

```sql
use AdventureWorksDW2022
go
select dp.ProductKey, dp.ProductAlternateKey, dp.EnglishProductName, dp.ListPrice
from   dbo.DimProduct as dp
where  dp.ProductAlternateKey like '[A-Z][A-Z]-[R,S][0-9][0-3]%';
```

**Reading the pattern from left to right**

| Piece | Meaning |
|---|---|
| `[A-Z]` | 1st character: a letter A-Z |
| `[A-Z]` | 2nd character: a letter A-Z |
| `-` | a literal hyphen |
| `[R,S]` | one character that is `R`, `S` (or a comma) |
| `[0-9]` | one digit |
| `[0-3]` | one digit from 0 to 3 |
| `%` | anything may follow |

Because the default collation is case-insensitive, `bk-r93r-62` would match as well.

**Which sample codes match?** (small Python table illustrating the rule)

In [36]:
show(
    """
    -- pattern: '[A-Z][A-Z]-[R,S][0-9][0-3]%'
    -- (sample codes tested; this is an illustration, not a query result)
    """,
    cols=['ProductAlternateKey', 'Matches'],
    rows=[
        ('BK-R93R-62', 'YES'),
        ('BK-R50B-44', 'YES'),
        ('BK-M18S-40', 'NO'),
        ('FR-R92B-58', 'YES'),
        ('FR-R38R-62', 'NO'),
        ('SO-B909-M', 'NO'),
        ('bk-r93r-62', 'YES'),
        ('B1-R93R-62', 'NO'),
    ]
)

ProductAlternateKey Matches
         BK-R93R-62     YES
         BK-R50B-44     YES
         BK-M18S-40      NO
         FR-R92B-58     YES
         FR-R38R-62      NO
          SO-B909-M      NO
         bk-r93r-62     YES
         B1-R93R-62      NO

(8 rows affected)


---
# Part 5 - Cheat Sheet

### Set operators

| Operator | Math | Duplicates | Typical question | Speed |
|---|---|---|---|---|
| `UNION ALL` | A + B | kept | "Stack everything" | fastest |
| `UNION` | A ∪ B | removed | "All distinct values from both" | slower |
| `INTERSECT` | A ∩ B | removed | "What is in both?" | slower |
| `EXCEPT` | A - B | removed | "In A but not in B?" | slower |
| `SELECT ... INTO` | - | - | "Save the result as a new table" | - |

### Filtering

| Need | Syntax |
|---|---|
| Compare | `=`  `<>`  `>`  `>=`  `<`  `<=` |
| Combine conditions | `AND`  `OR`  `NOT` (use parentheses) |
| Range (inclusive) | `col BETWEEN a AND b` / `NOT BETWEEN` |
| List of values | `col IN (a, b, c)` / `NOT IN` |
| Missing values | `col IS NULL` / `col IS NOT NULL` |
| Text pattern | `col LIKE 'a%'`  `'%a'`  `'%vi%'`  `'a_n%'`  `'[a-g]%'`  `'[^abc]%'` |

### Best practices

1. Match **column count and data types** across all set-operator queries; name columns in the **first** query.
2. Prefer `UNION ALL` when duplicates are impossible or acceptable.
3. Put `ORDER BY` only at the **end**.
4. Never use `= NULL`; use `IS NULL`.
5. Keep predicates **SARGable**: do not wrap the column in a function; compare it with a range or constant.
6. Be careful with `NOT IN` when NULLs may appear in the list.
7. Use parentheses when mixing `AND` and `OR`.